# Упражнения к семинару 13. Две выборки: t-тест, критерий Манна–Уитни, хи-квадрат для категориальных данных

Переименуйте файл в формате `Группа-Фамилия-Имя-seminar-13-tasks.ipynb`, например `2MP9-Ivanov-Ivan-seminar-13-tasks.ipynb`: этот файл сдаётся в Moodle. Формулировки упражнений здесь те же, что в листке `seminar-13-tasks.md`; дополнительно в ноутбуке заготовки функций, тесты и ячейки «Проверка».

Данные: таблицы Московской области за 2022–2025 годы, ДТП `datasets/legal/dtp-moscow-region.csv` и участники `datasets/legal/dtp-moscow-region-participants.csv`; путь в ячейках задан от папки `exercises/`, поэтому ноутбук должен лежать в ней, а папка `datasets` рядом, как в репозитории.

- Часть А: выполните ячейку с подключением библиотек, соберите таблицу водителей в упражнении А1, затем решайте упражнения по порядку и вводите числа и решения в тест Moodle «Семинар 13».
- Части Б и В: напишите тело функции вместо `pass`, запустите ячейку с тестами; если всё верно, она напечатает `Все тесты прошли успешно!`. Затем выполните ячейку «Проверка»: она вызывает функцию на таблицах области и печатает числа, которые вводятся в тест вместе с кодом функции.

Правила: функция не ссылается на переменные, определённые вне неё, кроме `pd`, `np`, `plt`, `sns`, `stats` и функций части Б; ячейки с тестами не меняются; в коде решений библиотеки, кроме pandas, NumPy, matplotlib, seaborn и scipy, не используются.

## Часть А. Разминка

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

sns.set_theme(style='whitegrid')       # оформление графиков по умолчанию
pd.set_option('display.width', 200)   # широкие таблицы печатаются без переноса

### А1. Стаж погибших и раненых водителей

Загрузите таблицу ДТП `datasets/legal/dtp-moscow-region.csv` в `df` с `index_col='id'` и `parse_dates=['datetime']` и таблицу участников `datasets/legal/dtp-moscow-region-participants.csv` в `people`. Соберите таблицу водителей с их ДТП, как в разделе 1 ноутбука занятия: стаж без кодов 95, 96 и 99 в новом столбце `people['experience_clean']`; `drivers = people[people['role'] == 'Водитель'].merge(df, left_on='dtp_id', right_index=True)` — должно получиться 25 364 водителя; возраст машины `drivers['vehicle_age']` — год ДТП минус `vehicle_year`, значения больше 100 — пропуск через `where`. Стаж погибших водителей без пропусков — `dead = drivers.loc[drivers['health'] == 'Погиб', 'experience_clean'].dropna()`, раненых — `injured`, так же со значением «Ранен». Каковы средние стажа двух групп, до сотых? Проверьте t-тестом Уэлча гипотезу «средний стаж погибших и раненых водителей одинаков» с двусторонней альтернативой и посчитайте d Коэна, погибшие — первая группа. Каковы t, до сотых, и d, до тысячных, и каково решение при 0,05? Подсказка: раздел 1 ноутбука занятия; `stats.ttest_ind(dead, injured, equal_var=False)`; d — разница средних, делённая на `np.sqrt((dead.var() + injured.var()) / 2)`. Без `equal_var=False` scipy считает t Стьюдента, и t получается другим. В выводе — оговорка: это связь, а не причина, опытные водители старше.

In [ ]:
# А1

### А2. То же критерием Манна–Уитни

Проверьте ту же гипотезу о стаже погибших и раненых водителей критерием Манна–Уитни, `stats.mannwhitneyu(dead, injured)`. Какова статистика U и каково решение при 0,05? Подсказка: раздел 2 ноутбука занятия; U считается для первой группы — в скольких парах «погибший, раненый» погибший опытнее, ничьи за половину пары; `stats.mannwhitneyu(injured, dead)` даёт другое U: n₁ · n₂ минус первое. p здесь меньше 0,001, и в тест вводятся U и решение. Сравните решение с решением t-теста из А1.

In [ ]:
# А2

### А3. Возраст машины у женщин и мужчин

Возраст машин водителей-женщин и водителей-мужчин без пропусков: `drivers.loc[drivers['gender'] == 'Женский', 'vehicle_age'].dropna()` и так же для «Мужской». Сколько значений в каждой группе, каковы средние, до сотых, и медианы? Проверьте t-тестом Уэлча и критерием Манна–Уитни гипотезу «возраст машин у женщин и мужчин одинаков» и посчитайте d Коэна, женщины — первая группа. Каковы p t-теста, до десятитысячных, p Манна–Уитни, до тысячных, d, до тысячных, и оба решения при 0,05? Каков размер эффекта по порогам d? Подсказка: раздел 2 ноутбука занятия; знак d — направление: минус, если среднее первой группы меньше; пороги d — 0,2, 0,5 и 0,8 по модулю, ниже 0,2 — ничтожный эффект. Если решения тестов расходятся, вспомните, о чём каждый: t — о средних, Манна–Уитни — о том, какая группа чаще больше. Вывод — о практической значимости: значимо не значит важно.

In [ ]:
# А3

### А4. Три исхода

Стаж водителей трёх исходов — погибших `dead`, раненых `injured` и не пострадавших, значение `health` «Не пострадал». Проверьте ANOVA Уэлча, `stats.f_oneway(..., equal_var=False)`, и критерием Крускала–Уоллиса, `stats.kruskal`, гипотезу «стаж одинаков при всех трёх исходах». Каковы F, до десятых, и H, до десятых, и каковы оба решения при 0,05? Подсказка: раздел 4 ноутбука занятия; три ряда без пропусков передаются через запятую: `stats.f_oneway(dead, injured, unhurt, equal_var=False)`; без `equal_var=False` scipy считает классическую ANOVA с общей дисперсией групп, и F получается другим; p меньше 0,001 в тест не вводятся. Отвергнутая гипотеза значит «не все равны»; какие пары групп различаются, отвечают попарные сравнения с поправкой — задача В1.

In [ ]:
# А4

### А5. Тёмное время и ДТП с погибшими

По таблице `df`: тёмное время суток — `df['light'].str.contains('темное')`, ДТП с погибшими — `df['dead_count'] > 0`. Постройте таблицу сопряжённости `pd.crosstab`, проверьте хи-квадратом независимости `stats.chi2_contingency` гипотезу «время суток и гибель людей независимы», посчитайте V Крамера и доли ДТП с погибшими в светлое и в тёмное время. Каковы χ², до десятых, V, до тысячных, обе доли, до тысячных, и решение при 0,05? Подсказка: раздел 5 ноутбука занятия; V = √(χ² / (n · (k − 1))), где χ² — статистика без поправки Йейтса, `stats.chi2_contingency(table, correction=False)`, n — сумма таблицы, k — меньшее из чисел строк и столбцов; то же V даёт `stats.contingency.association(table)`; доли — `pd.crosstab(..., normalize='index')`. Таблица долей вместо счётчиков даёт χ² 0 и p 1 без всякой ошибки; сами столбцы `df[['light', 'dead_count']]` вместо таблицы — `TypeError`.

In [ ]:
# А5

### А6. Погода и тяжесть ДТП

ДТП четырёх частых погод — «Ясно», «Пасмурно», «Дождь», «Снегопад» — и трёх тяжестей: `four = df[df['weather'].isin(['Ясно', 'Пасмурно', 'Дождь', 'Снегопад']) & (df['severity'] != 'Без пострадавших (нет данных)')]`. Постройте таблицу сопряжённости погоды и тяжести, проверьте хи-квадратом независимости гипотезу «погода и тяжесть ДТП независимы» и посчитайте V Крамера. Каково p, до стотысячных, и каково решение при 0,05? Каков V, до тысячных, и каков размер связи по порогам V? Подсказка: раздел 5 ноутбука занятия; пороги V — 0,1, 0,3 и 0,5, ниже 0,1 — ничтожная связь; наименьшее ожидаемое, `expected_freq.min()`, должно быть не меньше пяти, а редкие погоды дали бы малые ожидаемые. Сравните решение с размером связи: значимо — не значит важно.

In [ ]:
# А6

### А7. Районы в 2022 и 2025 годах и корреляция

Таблица ДТП по районам и годам — `pd.crosstab(df['district'], df['datetime'].dt.year)`; отберите районы, где в 2022 году не меньше 20 ДТП, и столбцы 2022 и 2025; строка «Московская область» — ДТП без указания округа, не район, — в 2022 году набирает меньше 20 ДТП, и отбор её не берёт. Сколько районов отобрано? Проверьте парным t-тестом и знаковым ранговым тестом Вилкоксона гипотезу «число ДТП в районах в 2022 и 2025 годах одинаково». Каковы p парного t-теста, до тысячных, и p теста Вилкоксона, до сотых, и оба решения при 0,05? Затем корреляция стажа и возраста машины у водителей, у которых известны оба значения: каков r Пирсона, до тысячных, и каково решение по его p? Подсказка: разделы 3 и 6 ноутбука занятия; `stats.ttest_rel(pairs[2022], pairs[2025])` и `stats.wilcoxon(pairs[2022], pairs[2025])`; пары водителей — `drivers[['experience_clean', 'vehicle_age']].dropna()`, затем `stats.pearsonr`. Две группировки по годам с отбором только 2022 года дают ряды разной длины и `ValueError`; `pearsonr` по столбцам с пропусками возвращает `nan`.

In [ ]:
# А7

## Часть Б. Основные задачи

### Б1. t-тест Уэлча и d Коэна

Первый инструмент для гипотезы 3 этапа 3: отличается ли средний срок рассмотрения в двух видах судов, средняя сумма иска в делах с адвокатом и без. Тест — `stats.ttest_ind` с `equal_var=False`, вариант Уэлча: он не предполагает равных дисперсий групп; по умолчанию scipy считает t Стьюдента, который на группах разного размера и разброса ошибается. Размер эффекта — d Коэна: разница средних, делённая на объединённое отклонение √((s₁² + s₂²) / 2), где s₁ и s₂ — стандартные отклонения групп с n − 1 в знаменателе, как у метода `std()`. Пропуски убираются заранее, иначе scipy вернёт `nan`.

Реализуйте функцию `t_two(a, b)`, которая сравнивает средние двух независимых групп t-тестом Уэлча и считает d Коэна. Аргументы: `a` — значения первой группы, Series чисел; пропуски допускаются; `b` — значения второй группы, Series чисел; пропуски допускаются. Возвращаемое значение: кортеж из трёх чисел (t, p, d): статистика и p-значение stats.ttest_ind(a, b, equal_var=False) по значениям без пропусков с двусторонней альтернативой и d Коэна — (среднее a − среднее b) / √((дисперсия a + дисперсия b) / 2). t и d больше нуля, если среднее a больше. Series не меняются.

Подсказка: раздел 1 ноутбука занятия: `a, b = a.dropna(), b.dropna()`, `result = stats.ttest_ind(a, b, equal_var=False)`, `d = (a.mean() - b.mean()) / np.sqrt((a.var() + b.var()) / 2)`. Без `equal_var=False` тест — Стьюдента, и на группах разного размера p другое; объединённое отклонение, взвешенное по размерам групп, — другая формула d, и на неравных группах число другое; `np.var` делит на n, а не на n − 1.

Проверка в ноутбуке: возраст машины водителей в ДТП с погибшими и без погибших через t_two; печатаются средние групп и t, p и d, в тест вводятся t до сотых, p до тысячных, d до тысячных и решение.

In [ ]:
def t_two(a, b):
    """
    Сравнивает средние двух независимых групп t-тестом Уэлча и считает d Коэна.

    Первый инструмент для гипотезы 3 этапа 3: отличается ли средний срок
    рассмотрения в двух видах судов, средняя сумма иска в делах с адвокатом и
    без. Тест — `stats.ttest_ind` с `equal_var=False`, вариант Уэлча: он не
    предполагает равных дисперсий групп; по умолчанию scipy считает t Стьюдента,
    который на группах разного размера и разброса ошибается. Размер эффекта — d
    Коэна: разница средних, делённая на объединённое отклонение √((s₁² + s₂²) /
    2), где s₁ и s₂ — стандартные отклонения групп с n − 1 в знаменателе, как у
    метода `std()`. Пропуски убираются заранее, иначе scipy вернёт `nan`.

    Аргументы:
        a: Значения первой группы, Series чисел; пропуски допускаются.
        b: Значения второй группы, Series чисел; пропуски допускаются.

    Возвращаемое значение:
        Кортеж из трёх чисел (t, p, d): статистика и p-значение
        stats.ttest_ind(a, b, equal_var=False) по значениям без пропусков с
        двусторонней альтернативой и d Коэна — (среднее a − среднее b) /
        √((дисперсия a + дисперсия b) / 2). t и d больше нуля, если среднее a
        больше. Series не меняются.
    """

    pass

In [ ]:
def t_two_tests():
    a = pd.Series([3.0, 9, 4, None, 12, 7, 15], index=list('abcdefg'))
    b = pd.Series([6.0, 5, 7, 6, None, 5, 8, 6, 7, 5, 6], index=list('hijklmnopqr'))
    before_a, before_b = a.copy(), b.copy()
    result = t_two(a, b)
    assert isinstance(result, tuple) and len(result) == 3, 'функция должна возвращать кортеж (t, p, d)'
    t, p, d = result
    assert round(t, 4) == 1.1647 and round(p, 4) == 0.2941, 't-тест Уэлча, equal_var=False, по значениям без пропусков: тест Стьюдента дал бы p 0,1556'
    assert round(d, 4) == 0.6665, 'd Коэна: разница средних, делённая на √((s₁² + s₂²) / 2)'
    t, p, d = t_two(b, a)
    assert round(t, 4) == -1.1647 and round(d, 4) == -0.6665, 'знак t и d — направление: минус, если среднее первой группы меньше'
    assert a.equals(before_a) and b.equals(before_b), 'Series изменились'

    print('Все тесты прошли успешно!')

In [ ]:
t_two_tests()

In [ ]:
# Проверка: возраст машины водителей в ДТП с погибшими и без погибших через t_two; печатаются средние групп и t, p и d, в тест вводятся t до сотых, p до тысячных, d до тысячных и решение

df_check = pd.read_csv('../datasets/legal/dtp-moscow-region.csv', index_col='id', parse_dates=['datetime'])
people_check = pd.read_csv('../datasets/legal/dtp-moscow-region-participants.csv')
people_check['experience_clean'] = people_check['experience'].replace([95, 96, 99], np.nan)
drivers_check = people_check[people_check['role'] == 'Водитель'].merge(df_check, left_on='dtp_id', right_index=True)
drivers_check['vehicle_age'] = drivers_check['datetime'].dt.year - drivers_check['vehicle_year']
drivers_check['vehicle_age'] = drivers_check['vehicle_age'].where(drivers_check['vehicle_age'] <= 100)

fatal = drivers_check['dead_count'] > 0
age = drivers_check['vehicle_age']
t, p, d = t_two(age[fatal], age[~fatal])
print(round(age[fatal].mean(), 2), round(age[~fatal].mean(), 2))
print(round(t, 2), round(p, 3), round(d, 3))

### Б2. Критерий Манна–Уитни

Непараметрический двойник t-теста — для скошенных данных с выбросами, малых групп и порядковых шкал вроде баллов и степеней тяжести. Критерий ставит значения обеих групп в один ряд по рангам; статистика U — число пар «значение из первой группы, значение из второй», в которых первое больше, равные значения — за половину пары. U считается для первой группы: если группы переставить, получится n₁ · n₂ − U. Пропуски убираются заранее, иначе scipy вернёт `nan`.

Реализуйте функцию `mw_two(a, b)`, которая сравнивает две независимые группы критерием Манна–Уитни. Аргументы: `a` — значения первой группы, Series чисел; пропуски допускаются; `b` — значения второй группы, Series чисел; пропуски допускаются. Возвращаемое значение: кортеж из двух чисел (U, p): статистика U первой группы и p-значение stats.mannwhitneyu с параметрами по умолчанию по значениям без пропусков. Series не меняются.

Подсказка: раздел 2 ноутбука занятия: `result = stats.mannwhitneyu(a.dropna(), b.dropna())`, затем поля `statistic` и `pvalue`. Меньшая из двух U, как в старых таблицах критерия, — не то, что возвращает scipy; `use_continuity=False` и `method='exact'` при равных значениях меняют p; односторонняя альтернатива уменьшает p вдвое.

Проверка в ноутбуке: возраст машины водителей в ДТП по выходным и в будни через mw_two; печатаются U, p и доля пар U / (n₁ · n₂), в тест вводятся p до тысячных, доля пар до тысячных и решение.

In [ ]:
def mw_two(a, b):
    """
    Сравнивает две независимые группы критерием Манна–Уитни.

    Непараметрический двойник t-теста — для скошенных данных с выбросами, малых
    групп и порядковых шкал вроде баллов и степеней тяжести. Критерий ставит
    значения обеих групп в один ряд по рангам; статистика U — число пар
    «значение из первой группы, значение из второй», в которых первое больше,
    равные значения — за половину пары. U считается для первой группы: если
    группы переставить, получится n₁ · n₂ − U. Пропуски убираются заранее, иначе
    scipy вернёт `nan`.

    Аргументы:
        a: Значения первой группы, Series чисел; пропуски допускаются.
        b: Значения второй группы, Series чисел; пропуски допускаются.

    Возвращаемое значение:
        Кортеж из двух чисел (U, p): статистика U первой группы и p-значение
        stats.mannwhitneyu с параметрами по умолчанию по значениям без
        пропусков. Series не меняются.
    """

    pass

In [ ]:
def mw_two_tests():
    a = pd.Series([3.0, 5, 5, 8, 9, None, 12])
    b = pd.Series([1.0, 2, 5, 6, 7, 7, None, 4])
    before_a, before_b = a.copy(), b.copy()
    result = mw_two(a, b)
    assert isinstance(result, tuple) and len(result) == 2, 'функция должна возвращать кортеж (U, p)'
    u, p = result
    assert u == 30 and round(p, 4) == 0.2214, 'критерий Манна–Уитни с параметрами по умолчанию по 6 и 7 значениям без пропусков; равные значения — за половину пары'
    u, p = mw_two(b, a)
    assert u == 12 and round(p, 4) == 0.2214, 'U — статистика первой группы: 6 · 7 − 30 = 12, p тот же'
    assert a.equals(before_a) and b.equals(before_b), 'Series изменились'

    print('Все тесты прошли успешно!')

In [ ]:
mw_two_tests()

In [ ]:
# Проверка: возраст машины водителей в ДТП по выходным и в будни через mw_two; печатаются U, p и доля пар U / (n₁ · n₂), в тест вводятся p до тысячных, доля пар до тысячных и решение

df_check = pd.read_csv('../datasets/legal/dtp-moscow-region.csv', index_col='id', parse_dates=['datetime'])
people_check = pd.read_csv('../datasets/legal/dtp-moscow-region-participants.csv')
people_check['experience_clean'] = people_check['experience'].replace([95, 96, 99], np.nan)
drivers_check = people_check[people_check['role'] == 'Водитель'].merge(df_check, left_on='dtp_id', right_index=True)
drivers_check['vehicle_age'] = drivers_check['datetime'].dt.year - drivers_check['vehicle_year']
drivers_check['vehicle_age'] = drivers_check['vehicle_age'].where(drivers_check['vehicle_age'] <= 100)

weekend = drivers_check['datetime'].dt.dayofweek >= 5
age = drivers_check['vehicle_age']
u, p = mw_two(age[weekend], age[~weekend])
print(u, round(p, 3), round(u / (age[weekend].count() * age[~weekend].count()), 3))

### Б3. Парный t-тест с выравниванием по ключу

Парные данные — одни и те же районы, суды или люди в два момента: число ДТП района в 2022 и 2025 годах, сроки суда до и после реформы. Таблица задана в длинном виде: в строке — объект, группа и значение, например район, год и число погибших. Пары собирает `pivot_table`: строка — объект, столбцы — группы; если у объекта в группе несколько строк, берётся их среднее, как у `pivot_table` по умолчанию. Объекты, у которых нет значения в одной из двух групп, в пары не входят. Тест — `stats.ttest_rel` по столбцам first и second выровненной таблицы.

Реализуйте функцию `paired_test(df, value, key, group, first, second)`, которая проверяет парным t-тестом, различаются ли значения одних и тех же объектов в двух группах. Аргументы: `df` — таблица в длинном виде со столбцами key, group и value; `value` — имя числового столбца, строка; `key` — имя столбца, по которому строятся пары, строка; `group` — имя столбца групп, строка; `first` — значение group, первая группа пары; `second` — значение group, вторая группа пары. Возвращаемое значение: кортеж из двух чисел (t, p): статистика и p-значение stats.ttest_rel по парам «значение в first, значение в second» для ключей, у которых есть значения в обеих группах; значение ключа в группе — среднее его строк, как у pivot_table по умолчанию. t больше нуля, если в first значения больше. Таблица df не меняется.

Подсказка: раздел 3 ноутбука занятия: `table = df.pivot_table(index=key, columns=group, values=value)`; `pairs = table[[first, second]].dropna()`; `stats.ttest_rel(pairs[first], pairs[second])`. Два отбора, `df[df[group] == first][value]` и `df[df[group] == second][value]`, не выровнены по ключу: ряды разной длины дают `ValueError`, а одинаковой — молча неверные пары; `dropna()` по всей таблице выбросит и ключи без значения в третьей группе; `pivot` вместо `pivot_table` падает на повторах ключа; `fillna(0)` вместо `dropna()` превращает отсутствие данных в ноль.

Проверка в ноутбуке: число погибших в ДТП по районам в 2022 и 2025 годах через paired_test по таблице в длинном виде; печатаются погибшие за каждый из двух лет по всей области, t и p, в тест вводятся t до сотых, p до тысячных и решение.

In [ ]:
def paired_test(df, value, key, group, first, second):
    """
    Проверяет парным t-тестом, различаются ли значения одних и тех же объектов в
    двух группах.

    Парные данные — одни и те же районы, суды или люди в два момента: число ДТП
    района в 2022 и 2025 годах, сроки суда до и после реформы. Таблица задана в
    длинном виде: в строке — объект, группа и значение, например район, год и
    число погибших. Пары собирает `pivot_table`: строка — объект, столбцы —
    группы; если у объекта в группе несколько строк, берётся их среднее, как у
    `pivot_table` по умолчанию. Объекты, у которых нет значения в одной из двух
    групп, в пары не входят. Тест — `stats.ttest_rel` по столбцам first и second
    выровненной таблицы.

    Аргументы:
        df: Таблица в длинном виде со столбцами key, group и value.
        value: Имя числового столбца, строка.
        key: Имя столбца, по которому строятся пары, строка.
        group: Имя столбца групп, строка.
        first: Значение group, первая группа пары.
        second: Значение group, вторая группа пары.

    Возвращаемое значение:
        Кортеж из двух чисел (t, p): статистика и p-значение stats.ttest_rel по
        парам «значение в first, значение в second» для ключей, у которых есть
        значения в обеих группах; значение ключа в группе — среднее его строк,
        как у pivot_table по умолчанию. t больше нуля, если в first значения
        больше. Таблица df не меняется.
    """

    pass

In [ ]:
def paired_test_tests():
    df = pd.DataFrame({
        'суд': ['В', 'А', 'Б', 'Г', 'Д', 'Б', 'А', 'В', 'Д', 'Е', 'Ж', 'Ж', 'А', 'А', 'Б', 'В'],
        'год': [2023, 2023, 2023, 2023, 2023, 2024, 2024, 2024, 2024, 2024, 2023, 2024, 2024, 2025, 2025, 2025],
        'дела': [60.0, 40, 25, 18, 33, 22, 31, 52, 35, 12, 47, None, 29, 20, 19, 41],
    })
    before = df.copy()
    result = paired_test(df, 'дела', 'суд', 'год', 2023, 2024)
    assert isinstance(result, tuple) and len(result) == 2, 'функция должна возвращать кортеж (t, p)'
    t, p = result
    assert round(t, 4) == 1.7666 and round(p, 4) == 0.1755, 'пары по суду: А, Б, В, Д; у Г и Е значение только в одной группе, у Ж в 2024 году пропуск; у суда А в 2024 году две строки — среднее 30'
    t, p = paired_test(df, 'дела', 'суд', 'год', 2024, 2023)
    assert round(t, 4) == -1.7666 and round(p, 4) == 0.1755, 'знак t: плюс, если в first значения больше'
    t, p = paired_test(df, 'дела', 'суд', 'год', 2023, 2025)
    assert round(t, 4) == 3.3265 and round(p, 4) == 0.0797, 'группы пары — first и second: в 2025 году значения есть у судов А, Б и В'
    assert df.equals(before), 'таблица изменилась'

    print('Все тесты прошли успешно!')

In [ ]:
paired_test_tests()

In [ ]:
# Проверка: число погибших в ДТП по районам в 2022 и 2025 годах через paired_test по таблице в длинном виде; печатаются погибшие за каждый из двух лет по всей области, t и p, в тест вводятся t до сотых, p до тысячных и решение

df_check = pd.read_csv('../datasets/legal/dtp-moscow-region.csv', index_col='id', parse_dates=['datetime'])

deaths = df_check.groupby(['district', df_check['datetime'].dt.year.rename('год')])['dead_count'].sum().reset_index()
t, p = paired_test(deaths, 'dead_count', 'district', 'год', 2022, 2025)
print(deaths.groupby('год')['dead_count'].sum().loc[[2022, 2025]].tolist())
print(round(t, 2), round(p, 3))

### Б4. ANOVA и Крускал–Уоллис

Больше двух групп сразу: стаж по трём исходам ДТП, срок рассмотрения в пяти судах. ANOVA Уэлча, `stats.f_oneway(..., equal_var=False)`, проверяет, что средние всех групп равны, и, как t Уэлча, не предполагает равных дисперсий групп; критерий Крускала–Уоллиса, `stats.kruskal`, — её непараметрический двойник по рангам. Группы — все значения столбца group в порядке `groupby`; строки с пропуском в group в группы не входят, пропуски в value убираются в каждой группе отдельно, а пропуски в других столбцах таблицы на тесты не влияют.

Реализуйте функцию `groups_test(df, value, group)`, которая сравнивает числовой столбец в нескольких группах ANOVA Уэлча и критерием Крускала–Уоллиса. Аргументы: `df` — таблица с числовым столбцом value и столбцом групп group; `value` — имя числового столбца, строка; `group` — имя столбца групп, строка. Возвращаемое значение: объект Series из четырёх чисел с индексом 'F', 'p ANOVA', 'H', 'p Крускала': статистика и p-значение stats.f_oneway с equal_var=False и статистика и p-значение stats.kruskal с параметрами по умолчанию по значениям value без пропусков в каждой группе group. Таблица df не меняется.

Подсказка: раздел 4 ноутбука занятия: группы — `[values.dropna() for name, values in df.groupby(group)[value]]`; функции получают их через звёздочку, `stats.f_oneway(*samples, equal_var=False)`: звёздочка раскладывает список на отдельные аргументы; ответ — `pd.Series({'F': ..., 'p ANOVA': ..., 'H': ..., 'p Крускала': ...})`. Без `dropna()` оба теста возвращают `nan`; `df.dropna()` по всей таблице выбрасывает и строки с пропусками в других столбцах — группы меньше, F другое; `groupby(..., dropna=False)` делает отдельную группу из пропуска; без `equal_var=False` — классическая ANOVA с общей дисперсией, другое F.

Проверка в ноутбуке: возраст машины водителей по тяжести ДТП через groups_test, без записи «Без пострадавших (нет данных)»; печатаются F, p ANOVA, H и p Крускала, в тест вводятся F до сотых, оба p до тысячных и оба решения.

In [ ]:
def groups_test(df, value, group):
    """
    Сравнивает числовой столбец в нескольких группах ANOVA Уэлча и критерием
    Крускала–Уоллиса.

    Больше двух групп сразу: стаж по трём исходам ДТП, срок рассмотрения в пяти
    судах. ANOVA Уэлча, `stats.f_oneway(..., equal_var=False)`, проверяет, что
    средние всех групп равны, и, как t Уэлча, не предполагает равных дисперсий
    групп; критерий Крускала–Уоллиса, `stats.kruskal`, — её непараметрический
    двойник по рангам. Группы — все значения столбца group в порядке `groupby`;
    строки с пропуском в group в группы не входят, пропуски в value убираются в
    каждой группе отдельно, а пропуски в других столбцах таблицы на тесты не
    влияют.

    Аргументы:
        df: Таблица с числовым столбцом value и столбцом групп group.
        value: Имя числового столбца, строка.
        group: Имя столбца групп, строка.

    Возвращаемое значение:
        Объект Series из четырёх чисел с индексом 'F', 'p ANOVA', 'H', 'p
        Крускала': статистика и p-значение stats.f_oneway с equal_var=False и
        статистика и p-значение stats.kruskal с параметрами по умолчанию по
        значениям value без пропусков в каждой группе group. Таблица df не
        меняется.
    """

    pass

In [ ]:
def groups_test_tests():
    df = pd.DataFrame({'район': ['Б', 'А', 'В', 'А', 'Б', 'В', 'А', None, 'Б', 'В', 'А', 'Б', 'В', 'В'],
                       'срок': [12.0, 8, 15, 9, None, 14, 7, 30, 11, 18, 10, 13, 16, None],
                       'сумма': [120.0, None, 85, 240, 60, None, 150, 90, 75, 310, None, 45, 200, 130]})
    before = df.copy()
    result = groups_test(df, 'срок', 'район')
    assert isinstance(result, pd.Series), 'функция должна возвращать Series'
    assert result.index.tolist() == ['F', 'p ANOVA', 'H', 'p Крускала'], "индекс 'F', 'p ANOVA', 'H', 'p Крускала', в этом порядке"
    assert result.round(4).tolist() == [20.9138, 0.0031, 8.9091, 0.0116], 'ANOVA Уэлча, equal_var=False, и Крускал–Уоллис по трём группам без пропусков срока; строка с пропуском в районе в группы не входит, а пропуски суммы на тесты не влияют; классическая ANOVA дала бы F 26,7071'
    assert groups_test(df[df['район'] != 'В'], 'срок', 'район').round(4).tolist() == [16.3333, 0.0101, 4.5, 0.0339], 'группы — все значения group, сколько их есть: здесь две'
    assert df.equals(before), 'таблица изменилась'

    print('Все тесты прошли успешно!')

In [ ]:
groups_test_tests()

In [ ]:
# Проверка: возраст машины водителей по тяжести ДТП через groups_test, без записи «Без пострадавших (нет данных)»; печатаются F, p ANOVA, H и p Крускала, в тест вводятся F до сотых, оба p до тысячных и оба решения

df_check = pd.read_csv('../datasets/legal/dtp-moscow-region.csv', index_col='id', parse_dates=['datetime'])
people_check = pd.read_csv('../datasets/legal/dtp-moscow-region-participants.csv')
people_check['experience_clean'] = people_check['experience'].replace([95, 96, 99], np.nan)
drivers_check = people_check[people_check['role'] == 'Водитель'].merge(df_check, left_on='dtp_id', right_index=True)
drivers_check['vehicle_age'] = drivers_check['datetime'].dt.year - drivers_check['vehicle_year']
drivers_check['vehicle_age'] = drivers_check['vehicle_age'].where(drivers_check['vehicle_age'] <= 100)

three = drivers_check[drivers_check['severity'] != 'Без пострадавших (нет данных)']
print(groups_test(three, 'vehicle_age', 'severity').round(3))

### Б5. Хи-квадрат независимости и V Крамера

Инструмент гипотезы 4 этапа 3: связаны ли вид спора и исход дела, район и гибель людей. Таблица сопряжённости — `pd.crosstab(df[a], df[b])`: строки с пропуском в одном из столбцов в неё не входят. Тест — `stats.chi2_contingency` с параметрами по умолчанию; у таблицы 2 × 2 он вносит поправку Йейтса, и χ² и p возвращаются с ней. V Крамера — √(χ² / (n · (k − 1))), где χ² — статистика без поправки Йейтса, как в учебниках, n — сумма таблицы, k — меньшее из чисел строк и столбцов; то же V считает `stats.contingency.association`. У таблиц больше 2 × 2 поправки нет, и χ² для p и для V один и тот же.

Реализуйте функцию `chi_square(df, a, b)`, которая проверяет хи-квадратом независимости, связаны ли два категориальных столбца, и считает V Крамера. Аргументы: `df` — таблица с категориальными столбцами a и b; `a` — имя первого столбца, строка; `b` — имя второго столбца, строка. Возвращаемое значение: кортеж из четырёх чисел (χ², p, dof, V): статистика, p-значение и число степеней свободы stats.chi2_contingency с параметрами по умолчанию по таблице сопряжённости pd.crosstab(df[a], df[b]) и V Крамера по χ² той же таблицы без поправки Йейтса, correction=False. Таблица df не меняется.

Подсказка: раздел 5 ноутбука занятия: `table = pd.crosstab(df[a], df[b])`; `result = stats.chi2_contingency(table)`; `plain = stats.chi2_contingency(table, correction=False)`; `v = np.sqrt(plain.statistic / (table.to_numpy().sum() * (min(table.shape) - 1)))`. Сами столбцы `df[[a, b]]` вместо таблицы сопряжённости дают `TypeError`; таблица долей, `normalize='index'`, — χ² около нуля без ошибки; `correction=False` для χ² и p у таблицы 2 × 2 даёт χ² больше, а V по χ² с поправкой — V меньше; n — сумма таблицы, а не `len(df)`: строки с пропусками в таблицу не входят.

Проверка в ноутбуке: пол водителя и исход ДТП для водителя через chi_square и доли погибших по полу; печатаются χ², p, число степеней свободы, V и доли погибших, в тест вводятся χ² до десятых, V до тысячных, доля погибших среди женщин до тысячных и решение.

In [ ]:
def chi_square(df, a, b):
    """
    Проверяет хи-квадратом независимости, связаны ли два категориальных столбца,
    и считает V Крамера.

    Инструмент гипотезы 4 этапа 3: связаны ли вид спора и исход дела, район и
    гибель людей. Таблица сопряжённости — `pd.crosstab(df[a], df[b])`: строки с
    пропуском в одном из столбцов в неё не входят. Тест —
    `stats.chi2_contingency` с параметрами по умолчанию; у таблицы 2 × 2 он
    вносит поправку Йейтса, и χ² и p возвращаются с ней. V Крамера — √(χ² / (n ·
    (k − 1))), где χ² — статистика без поправки Йейтса, как в учебниках, n —
    сумма таблицы, k — меньшее из чисел строк и столбцов; то же V считает
    `stats.contingency.association`. У таблиц больше 2 × 2 поправки нет, и χ²
    для p и для V один и тот же.

    Аргументы:
        df: Таблица с категориальными столбцами a и b.
        a: Имя первого столбца, строка.
        b: Имя второго столбца, строка.

    Возвращаемое значение:
        Кортеж из четырёх чисел (χ², p, dof, V): статистика, p-значение и число
        степеней свободы stats.chi2_contingency с параметрами по умолчанию по
        таблице сопряжённости pd.crosstab(df[a], df[b]) и V Крамера по χ² той же
        таблицы без поправки Йейтса, correction=False. Таблица df не меняется.
    """

    pass

In [ ]:
def chi_square_tests():
    df = pd.DataFrame({
        'суд': ['А', 'Б', 'А', 'А', 'Б', 'Б', 'А', 'Б', None, 'А', 'Б', 'А', 'Б', 'А', 'Б', 'А', 'Б', 'Б', 'А', 'А'],
        'исход': ['иск', 'отказ', 'иск', 'мир', 'отказ', 'иск', 'иск', 'отказ', 'иск', 'мир', 'отказ', 'иск', 'мир', 'иск', 'отказ', 'отказ', None, 'отказ', 'иск', 'иск'],
        'адвокат': [True, False, True, True, False, False, True, False, True, True, False, True, True, False, False, True, True, False, True, False],
    })
    before = df.copy()
    result = chi_square(df, 'суд', 'исход')
    assert isinstance(result, tuple) and len(result) == 4, 'функция должна возвращать кортеж (χ², p, dof, V)'
    chi2, p, dof, v = result
    assert round(chi2, 4) == 8.2848 and round(p, 4) == 0.0159 and dof == 2, 'хи-квадрат по таблице сопряжённости 2 × 3; строки с пропуском в таблицу не входят'
    assert round(v, 4) == 0.6784, 'V Крамера: √(χ² / (n · (k − 1))), n = 18 — сумма таблицы, k = 2'
    chi2, p, dof, v = chi_square(df, 'суд', 'адвокат')
    assert round(chi2, 4) == 4.2369 and round(p, 4) == 0.0396 and dof == 1, 'таблица 2 × 2: χ² и p — chi2_contingency по умолчанию, с поправкой Йейтса; без неё χ² был бы 6,3427'
    assert round(v, 4) == 0.5778, 'V по χ² без поправки Йейтса: √(6,3427 / 19); по χ² с поправкой вышло бы 0,4722'
    assert df.equals(before), 'таблица изменилась'

    print('Все тесты прошли успешно!')

In [ ]:
chi_square_tests()

In [ ]:
# Проверка: пол водителя и исход ДТП для водителя через chi_square и доли погибших по полу; печатаются χ², p, число степеней свободы, V и доли погибших, в тест вводятся χ² до десятых, V до тысячных, доля погибших среди женщин до тысячных и решение

df_check = pd.read_csv('../datasets/legal/dtp-moscow-region.csv', index_col='id', parse_dates=['datetime'])
people_check = pd.read_csv('../datasets/legal/dtp-moscow-region-participants.csv')
people_check['experience_clean'] = people_check['experience'].replace([95, 96, 99], np.nan)
drivers_check = people_check[people_check['role'] == 'Водитель'].merge(df_check, left_on='dtp_id', right_index=True)
drivers_check['vehicle_age'] = drivers_check['datetime'].dt.year - drivers_check['vehicle_year']
drivers_check['vehicle_age'] = drivers_check['vehicle_age'].where(drivers_check['vehicle_age'] <= 100)

chi2, p, dof, v = chi_square(drivers_check, 'gender', 'health')
print(round(chi2, 1), p, dof, round(v, 3))
print(pd.crosstab(drivers_check['gender'], drivers_check['health'], normalize='index')['Погиб'].round(3))

### Б6. Точный тест Фишера для таблицы 2 × 2

Хи-квадрат опирается на приближение, которое неточно, когда ожидаемых в клетке меньше пяти: в маленьком суде, районе, за короткий период. Для таблицы 2 × 2 есть точный тест Фишера, `stats.fisher_exact`: p считается перебором всех таблиц с теми же суммами строк и столбцов. scipy 1.18 считает тест и для больших таблиц — по случайным таблицам, и p там меняется от запуска к запуску, — но в курсе он нужен для 2 × 2; если в столбце больше двух значений, функция поднимает ValueError: такие таблицы проверяют хи-квадратом, объединив редкие категории.

Реализуйте функцию `fisher_2x2(df, a, b)`, которая проверяет точным тестом Фишера связь двух столбцов, у каждого из которых два значения. Аргументы: `df` — таблица со столбцами a и b; `a` — имя первого столбца с двумя значениями, строка; `b` — имя второго столбца с двумя значениями, строка. Возвращаемое значение: число — p-значение stats.fisher_exact с двусторонней альтернативой по таблице сопряжённости pd.crosstab(df[a], df[b]); строки с пропуском в таблицу не входят. Если таблица не 2 × 2 — ValueError. Таблица df не меняется.

Подсказка: раздел 5 ноутбука занятия: `table = pd.crosstab(df[a], df[b])`; `if table.shape != (2, 2): raise ValueError(...)`; `stats.fisher_exact(table).pvalue`. Поле `statistic` — отношение шансов, а не p; без проверки размера scipy молча считает тест и для таблицы 3 × 2; p хи-квадрата на малой таблице — приближение, и оно другое.

Проверка в ноутбуке: тёмное время и ДТП с погибшими в Зарайском районе через fisher_2x2 и для сравнения хи-квадратом; печатаются таблица, наименьшее ожидаемое, p теста Фишера и p хи-квадрата, в тест вводятся оба p до тысячных и решение теста Фишера.

In [ ]:
def fisher_2x2(df, a, b):
    """
    Проверяет точным тестом Фишера связь двух столбцов, у каждого из которых два
    значения.

    Хи-квадрат опирается на приближение, которое неточно, когда ожидаемых в
    клетке меньше пяти: в маленьком суде, районе, за короткий период. Для
    таблицы 2 × 2 есть точный тест Фишера, `stats.fisher_exact`: p считается
    перебором всех таблиц с теми же суммами строк и столбцов. scipy 1.18 считает
    тест и для больших таблиц — по случайным таблицам, и p там меняется от
    запуска к запуску, — но в курсе он нужен для 2 × 2; если в столбце больше
    двух значений, функция поднимает ValueError: такие таблицы проверяют хи-
    квадратом, объединив редкие категории.

    Аргументы:
        df: Таблица со столбцами a и b.
        a: Имя первого столбца с двумя значениями, строка.
        b: Имя второго столбца с двумя значениями, строка.

    Возвращаемое значение:
        Число — p-значение stats.fisher_exact с двусторонней альтернативой по
        таблице сопряжённости pd.crosstab(df[a], df[b]); строки с пропуском в
        таблицу не входят. Если таблица не 2 × 2 — ValueError. Таблица df не
        меняется.
    """

    pass

In [ ]:
def fisher_2x2_tests():
    df = pd.DataFrame({
        'ночь': [True, False, False, True, False, False, True, False, False, False, True, False, None, False, True, False, True, False],
        'погибшие': [True, False, False, True, False, True, False, False, True, False, True, False, True, False, False, False, False, False],
        'погода': ['ясно', 'дождь', 'ясно', 'снег', 'ясно', 'дождь', 'ясно', 'ясно', 'снег', 'ясно', 'дождь', 'ясно', 'ясно', 'дождь', 'ясно', 'снег', 'ясно', 'дождь'],
    })
    before = df.copy()
    p = fisher_2x2(df, 'ночь', 'погибшие')
    assert isinstance(p, float), 'функция должна возвращать число — p-значение'
    assert round(p, 4) == 0.2801, 'точный тест Фишера, альтернатива двусторонняя, по таблице 2 × 2 без строки с пропуском; хи-квадрат дал бы 0,4128'
    assert round(fisher_2x2(df, 'погибшие', 'ночь'), 4) == 0.2801, 'перестановка столбцов не меняет p'
    for a, b in [('погода', 'погибшие'), ('погибшие', 'погода')]:
        try:
            fisher_2x2(df, a, b)
        except ValueError:
            pass
        else:
            raise AssertionError(f'таблица {a} × {b} — не 2 × 2: должно быть ValueError')
    assert df.equals(before), 'таблица изменилась'

    print('Все тесты прошли успешно!')

In [ ]:
fisher_2x2_tests()

In [ ]:
# Проверка: тёмное время и ДТП с погибшими в Зарайском районе через fisher_2x2 и для сравнения хи-квадратом; печатаются таблица, наименьшее ожидаемое, p теста Фишера и p хи-квадрата, в тест вводятся оба p до тысячных и решение теста Фишера

df_check = pd.read_csv('../datasets/legal/dtp-moscow-region.csv', index_col='id', parse_dates=['datetime'])

town = df_check[df_check['district'] == 'Зарайский'].copy()
town['dark'] = town['light'].str.contains('темное')
town['fatal'] = town['dead_count'] > 0
table = pd.crosstab(town['dark'], town['fatal'])
chi = stats.chi2_contingency(table)
print(table.to_numpy().tolist(), round(chi.expected_freq.min(), 2))
print(round(fisher_2x2(town, 'dark', 'fatal'), 3), round(chi.pvalue, 3))

### Б7. Значимость корреляции

Связаны ли два числовых столбца: сумма иска и срок рассмотрения, стаж и возраст машины. `stats.pearsonr` проверяет линейную связь, `stats.spearmanr` — монотонную, по рангам и устойчиво к выбросам; нулевая гипотеза у обоих — связи нет. Обе функции пропусков не пропускают, поэтому пары берутся там, где известны оба значения: отбор по двум столбцам сразу, а не `dropna()` у каждого по отдельности — иначе пары разъедутся.

Реализуйте функцию `corr_test(x, y)`, которая проверяет значимость корреляции Пирсона и Спирмена по парам без пропусков. Аргументы: `x` — первый числовой столбец, Series; пропуски допускаются; `y` — второй числовой столбец, Series с тем же индексом; пропуски допускаются. Возвращаемое значение: кортеж из четырёх чисел (r, p, ρ, p): коэффициент и p-значение stats.pearsonr и коэффициент и p-значение stats.spearmanr, оба с двусторонней альтернативой, по парам, где известны и x, и y. Series не меняются.

Подсказка: раздел 6 ноутбука занятия: `both = x.notna() & y.notna()`; `stats.pearsonr(x[both], y[both])` и `stats.spearmanr(x[both], y[both])`. `x.dropna()` и `y.dropna()` по отдельности не совпадают по строкам и при одинаковой длине дают молча неверные пары; `stats.kendalltau` — другой коэффициент по рангам; метод pandas `x.corr(y)` даёт r, но не p.

Проверка в ноутбуке: стаж и возраст машины водителей Подольского района через corr_test; печатаются число пар и r и p Пирсона, ρ и p Спирмена, в тест вводятся оба коэффициента и оба p до тысячных и оба решения.

In [ ]:
def corr_test(x, y):
    """
    Проверяет значимость корреляции Пирсона и Спирмена по парам без пропусков.

    Связаны ли два числовых столбца: сумма иска и срок рассмотрения, стаж и
    возраст машины. `stats.pearsonr` проверяет линейную связь, `stats.spearmanr`
    — монотонную, по рангам и устойчиво к выбросам; нулевая гипотеза у обоих —
    связи нет. Обе функции пропусков не пропускают, поэтому пары берутся там,
    где известны оба значения: отбор по двум столбцам сразу, а не `dropna()` у
    каждого по отдельности — иначе пары разъедутся.

    Аргументы:
        x: Первый числовой столбец, Series; пропуски допускаются.
        y: Второй числовой столбец, Series с тем же индексом; пропуски
           допускаются.

    Возвращаемое значение:
        Кортеж из четырёх чисел (r, p, ρ, p): коэффициент и p-значение
        stats.pearsonr и коэффициент и p-значение stats.spearmanr, оба с
        двусторонней альтернативой, по парам, где известны и x, и y. Series не
        меняются.
    """

    pass

In [ ]:
def corr_test_tests():
    x = pd.Series([2.0, 4, None, 7, 5, 9, 11, 3, None, 8, 12, 6, 10], index=list('abcdefghijklm'))
    y = pd.Series([4.0, 3, 2, None, 6, 5, 7, 2, 6, None, 9, 8, 3], index=list('abcdefghijklm'))
    before_x, before_y = x.copy(), y.copy()
    result = corr_test(x, y)
    assert isinstance(result, tuple) and len(result) == 4, 'функция должна возвращать кортеж (r, p, ρ, p)'
    r, p_r, rho, p_rho = result
    assert round(r, 4) == 0.559 and round(p_r, 4) == 0.1177, 'корреляция Пирсона по девяти парам, где известны оба значения'
    assert round(rho, 4) == 0.6193 and round(p_rho, 4) == 0.0753, 'корреляция Спирмена по тем же девяти парам'
    assert [round(v, 4) for v in corr_test(y, x)] == [0.559, 0.1177, 0.6193, 0.0753], 'корреляция симметрична: пары те же при любом порядке столбцов'
    assert x.equals(before_x) and y.equals(before_y), 'Series изменились'

    print('Все тесты прошли успешно!')

In [ ]:
corr_test_tests()

In [ ]:
# Проверка: стаж и возраст машины водителей Подольского района через corr_test; печатаются число пар и r и p Пирсона, ρ и p Спирмена, в тест вводятся оба коэффициента и оба p до тысячных и оба решения

df_check = pd.read_csv('../datasets/legal/dtp-moscow-region.csv', index_col='id', parse_dates=['datetime'])
people_check = pd.read_csv('../datasets/legal/dtp-moscow-region-participants.csv')
people_check['experience_clean'] = people_check['experience'].replace([95, 96, 99], np.nan)
drivers_check = people_check[people_check['role'] == 'Водитель'].merge(df_check, left_on='dtp_id', right_index=True)
drivers_check['vehicle_age'] = drivers_check['datetime'].dt.year - drivers_check['vehicle_year']
drivers_check['vehicle_age'] = drivers_check['vehicle_age'].where(drivers_check['vehicle_age'] <= 100)

town = drivers_check[drivers_check['district'] == 'Подольский']
r, p_r, rho, p_rho = corr_test(town['experience_clean'], town['vehicle_age'])
print((town['experience_clean'].notna() & town['vehicle_age'].notna()).sum())
print(round(r, 3), round(p_r, 3), round(rho, 3), round(p_rho, 3))

### Б8. Размер эффекта словами

Рядом с p в выводе пишут размер эффекта, и не только числом. Пороги: для d Коэна — 0,2, 0,5 и 0,8; для V Крамера и коэффициентов корреляции r и ρ — 0,1, 0,3 и 0,5. Значение сравнивается по модулю: d −0,6 и 0,6 — одинаково средние; значение ровно на пороге относится к большему размеру; ниже первого порога эффект ничтожный. Пороги условны — их предложил Коэн для психологии, — поэтому в выводе юриста рядом со словом стоит сама разница. Размер эффекта `nan` значит пропуски в данных, и функция поднимает ValueError, а не пишет слово.

Реализуйте функцию `effect_label(value, kind)`, которая называет размер эффекта словом по порогам Коэна. Аргументы: `value` — размер эффекта, число; `kind` — мера, строка 'd', 'V' или 'r'. Возвращаемое значение: строка «ничтожный», если |value| меньше первого порога меры kind, «малый» — не меньше первого, «средний» — не меньше второго, «большой» — не меньше третьего. Если kind не 'd', 'V' или 'r' или value — nan, ValueError.

Подсказка: словарь порогов внутри функции, `{'d': (0.2, 0.5, 0.8), 'V': (0.1, 0.3, 0.5), 'r': (0.1, 0.3, 0.5)}`; проверки `kind not in thresholds` и `np.isnan(value)` — в начале; затем `abs(value)` сравнивается с порогами от большего к меньшему через `>=`. Без `abs` отрицательный d всегда «ничтожный»; `>` вместо `>=` относит значение на пороге к меньшему размеру; пороги d для V и r занижают связь.

Проверка в ноутбуке: три размера эффекта по таблицам области через effect_label — d Коэна стажа женщин и мужчин, r Пирсона и ρ Спирмена числа участников и числа раненых в ДТП; печатаются значения и слова, в тест вводятся d до тысячных и все три слова.

In [ ]:
def effect_label(value, kind):
    """
    Называет размер эффекта словом по порогам Коэна.

    Рядом с p в выводе пишут размер эффекта, и не только числом. Пороги: для d
    Коэна — 0,2, 0,5 и 0,8; для V Крамера и коэффициентов корреляции r и ρ —
    0,1, 0,3 и 0,5. Значение сравнивается по модулю: d −0,6 и 0,6 — одинаково
    средние; значение ровно на пороге относится к большему размеру; ниже первого
    порога эффект ничтожный. Пороги условны — их предложил Коэн для психологии,
    — поэтому в выводе юриста рядом со словом стоит сама разница. Размер эффекта
    `nan` значит пропуски в данных, и функция поднимает ValueError, а не пишет
    слово.

    Аргументы:
        value: Размер эффекта, число.
        kind: Мера, строка 'd', 'V' или 'r'.

    Возвращаемое значение:
        Строка «ничтожный», если |value| меньше первого порога меры kind,
        «малый» — не меньше первого, «средний» — не меньше второго, «большой» —
        не меньше третьего. Если kind не 'd', 'V' или 'r' или value — nan,
        ValueError.
    """

    pass

In [ ]:
def effect_label_tests():
    assert effect_label(0.1, 'd') == 'ничтожный', 'd меньше 0,2 — ничтожный'
    assert effect_label(0.2, 'd') == 'малый' and effect_label(0.49, 'd') == 'малый', 'd от 0,2 до 0,5 — малый; значение на пороге — больший размер'
    assert effect_label(0.5, 'd') == 'средний' and effect_label(-0.8, 'd') == 'большой', 'пороги d — 0,5 и 0,8; d сравнивается по модулю'
    assert effect_label(1.7, 'd') == 'большой', 'd бывает и больше единицы'
    assert effect_label(0.05, 'V') == 'ничтожный' and effect_label(0.15, 'V') == 'малый', 'V: порог малой связи — 0,1, а не 0,2'
    assert effect_label(0.3, 'V') == 'средний' and effect_label(0.5, 'V') == 'большой', 'пороги V — 0,3 и 0,5'
    assert effect_label(-0.35, 'r') == 'средний' and effect_label(0.6, 'r') == 'большой', 'r по модулю; пороги r — 0,1, 0,3 и 0,5'
    assert effect_label(np.float64(0.099), 'r') == 'ничтожный', 'число NumPy тоже годится'
    for bad in [(0.3, 'eta'), (float('nan'), 'd')]:
        try:
            effect_label(*bad)
        except ValueError:
            pass
        else:
            raise AssertionError(f'effect_label{bad} должно давать ValueError')

    print('Все тесты прошли успешно!')

In [ ]:
effect_label_tests()

In [ ]:
# Проверка: три размера эффекта по таблицам области через effect_label — d Коэна стажа женщин и мужчин, r Пирсона и ρ Спирмена числа участников и числа раненых в ДТП; печатаются значения и слова, в тест вводятся d до тысячных и все три слова

df_check = pd.read_csv('../datasets/legal/dtp-moscow-region.csv', index_col='id', parse_dates=['datetime'])
people_check = pd.read_csv('../datasets/legal/dtp-moscow-region-participants.csv')
people_check['experience_clean'] = people_check['experience'].replace([95, 96, 99], np.nan)
drivers_check = people_check[people_check['role'] == 'Водитель'].merge(df_check, left_on='dtp_id', right_index=True)
drivers_check['vehicle_age'] = drivers_check['datetime'].dt.year - drivers_check['vehicle_year']
drivers_check['vehicle_age'] = drivers_check['vehicle_age'].where(drivers_check['vehicle_age'] <= 100)

experience = drivers_check['experience_clean']
women = experience[drivers_check['gender'] == 'Женский'].dropna()
men = experience[drivers_check['gender'] == 'Мужской'].dropna()
d = (women.mean() - men.mean()) / np.sqrt((women.var() + men.var()) / 2)
r = df_check['participants_count'].corr(df_check['injured_count'])
rho = df_check['participants_count'].corr(df_check['injured_count'], method='spearman')
print(round(d, 3), effect_label(d, 'd'))
print(round(r, 3), effect_label(r, 'r'))
print(round(rho, 3), effect_label(rho, 'r'))

## Часть В. Задачи повышенной сложности

### В1. Попарные сравнения с поправкой

ANOVA, отвергнув «все средние равны», не говорит, какие группы различаются. Ответ — попарные сравнения: каждая пара групп — t_two из Б1, то есть t Уэлча, p и d Коэна. Пар при k группах k · (k − 1) / 2, и поправка Бонферрони делит alpha на число пар, а не групп. Группы — значения столбца group в порядке `groupby`, строки с пропуском в group в группы не входят; пары — в порядке групп: первая со второй, первая с третьей и так далее, затем вторая с третьей.

Реализуйте функцию `compare_groups(df, value, group, alpha=0.05)`, которая сравнивает все пары групп t-тестом Уэлча с поправкой Бонферрони. Аргументы: `df` — таблица с числовым столбцом value и столбцом групп group; `value` — имя числового столбца, строка; `group` — имя столбца групп, строка; `alpha` — уровень значимости на всё семейство сравнений, число; по умолчанию 0.05. Возвращаемое значение: объект DataFrame: строка на пару групп в указанном порядке, индекс 0, 1, 2, …; столбцы «группа 1» и «группа 2» — названия групп пары, «t», «p», «d» — результат t_two(значения первой группы, значения второй) и «значимо после поправки» — p < alpha / число пар. Таблица df не меняется.

Подсказка: группы — словарь `{name: values for name, values in df.groupby(group)[value]}`, названия — `list(...)` от него; пары — двойной цикл `for i in range(len(names)):` и `for j in range(i + 1, len(names)):`; строки — список словарей, таблица — `pd.DataFrame(rows)`, поправка — `table['p'] < alpha / len(table)`. Деление на число групп вместо числа пар — поправка слабее нужной; внутренний цикл `range(len(names))` берёт каждую пару дважды и пары группы с самой собой.

Проверка в ноутбуке: попарные сравнения возраста машины водителей по тяжести ДТП через compare_groups, без записи «Без пострадавших (нет данных)»; печатается таблица пар, в тест вводятся число пар, значимых после поправки, p пары «Легкий» — «С погибшими» до тысячных и пара с наибольшим |d|.

In [ ]:
def compare_groups(df, value, group, alpha=0.05):
    """
    Сравнивает все пары групп t-тестом Уэлча с поправкой Бонферрони.

    ANOVA, отвергнув «все средние равны», не говорит, какие группы различаются.
    Ответ — попарные сравнения: каждая пара групп — t_two из Б1, то есть t
    Уэлча, p и d Коэна. Пар при k группах k · (k − 1) / 2, и поправка Бонферрони
    делит alpha на число пар, а не групп. Группы — значения столбца group в
    порядке `groupby`, строки с пропуском в group в группы не входят; пары — в
    порядке групп: первая со второй, первая с третьей и так далее, затем вторая
    с третьей.

    Аргументы:
        df: Таблица с числовым столбцом value и столбцом групп group.
        value: Имя числового столбца, строка.
        group: Имя столбца групп, строка.
        alpha: Уровень значимости на всё семейство сравнений, число; по
               умолчанию 0.05.

    Возвращаемое значение:
        Объект DataFrame: строка на пару групп в указанном порядке, индекс 0, 1,
        2, …; столбцы «группа 1» и «группа 2» — названия групп пары, «t», «p»,
        «d» — результат t_two(значения первой группы, значения второй) и
        «значимо после поправки» — p < alpha / число пар. Таблица df не
        меняется.
    """

    pass

In [ ]:
def compare_groups_tests():
    df = pd.DataFrame({
        'район': ['В', 'А', 'Б', 'Г', 'В', 'А', 'Б', 'Г', 'А', 'В', 'А', 'Б', 'Г', None, 'В', 'А', 'Б', 'Г', 'В', 'А', 'Б', 'Г', 'Б', 'Г', 'В'],
        'срок': [12.0, 8, 16, 10, 9, 10, 15, 11, None, 13, 11, 15, 8, 40, 11, 10, 15, 9, 15, 3, 15, 11, 15, 5, None],
        'судья': ['Орлова', None, 'Белов', 'Орлова', 'Белов', 'Орлова', None, 'Белов', 'Орлова', 'Белов', 'Орлова', 'Белов', None,
                  'Белов', 'Орлова', 'Белов', 'Орлова', 'Белов', None, 'Орлова', 'Белов', 'Орлова', 'Белов', 'Орлова', 'Белов'],
    })
    before = df.copy()
    table = compare_groups(df, 'срок', 'район')
    assert isinstance(table, pd.DataFrame), 'функция должна возвращать DataFrame'
    assert table.columns.tolist() == ['группа 1', 'группа 2', 't', 'p', 'd', 'значимо после поправки'], 'столбцы как в условии, в том же порядке'
    assert table.index.tolist() == [0, 1, 2, 3, 4, 5], 'шесть пар четырёх групп, индекс с нуля; строка с пропуском в районе в группы не входит'
    assert list(zip(table['группа 1'], table['группа 2'])) == [('А', 'Б'), ('А', 'В'), ('А', 'Г'), ('Б', 'В'), ('Б', 'Г'), ('В', 'Г')], 'пары в порядке групп groupby: А с Б, А с В, А с Г, Б с В, …'
    assert table['p'].round(4).tolist() == [0.0088, 0.0778, 0.736, 0.0329, 0.001, 0.0567], 'p — t-тест Уэлча каждой пары по значениям срока без пропусков; пропуски в столбце судьи на сравнения не влияют'
    assert table['t'].round(4).tolist() == [-4.6831, -2.058, -0.3507, 3.1236, 6.5204, 2.1958], 't — t_two(первая группа пары, вторая)'
    assert table['d'].round(4).tolist() == [-2.9579, -1.3016, -0.2155, 1.9702, 3.7646, 1.3284], 'd — d Коэна из t_two'
    assert table['значимо после поправки'].tolist() == [False, False, False, False, True, False], 'поправка Бонферрони на шесть пар, порог 0,05 / 6; при делении на четыре группы p 0,0088 было бы значимо'
    assert compare_groups(df, 'срок', 'район', alpha=0.1)['значимо после поправки'].tolist() == [True, False, False, False, True, False], 'уровень значимости alpha: порог 0,1 / 6'
    assert df.equals(before), 'таблица изменилась'

    print('Все тесты прошли успешно!')

In [ ]:
compare_groups_tests()

In [ ]:
# Проверка: попарные сравнения возраста машины водителей по тяжести ДТП через compare_groups, без записи «Без пострадавших (нет данных)»; печатается таблица пар, в тест вводятся число пар, значимых после поправки, p пары «Легкий» — «С погибшими» до тысячных и пара с наибольшим |d|

df_check = pd.read_csv('../datasets/legal/dtp-moscow-region.csv', index_col='id', parse_dates=['datetime'])
people_check = pd.read_csv('../datasets/legal/dtp-moscow-region-participants.csv')
people_check['experience_clean'] = people_check['experience'].replace([95, 96, 99], np.nan)
drivers_check = people_check[people_check['role'] == 'Водитель'].merge(df_check, left_on='dtp_id', right_index=True)
drivers_check['vehicle_age'] = drivers_check['datetime'].dt.year - drivers_check['vehicle_year']
drivers_check['vehicle_age'] = drivers_check['vehicle_age'].where(drivers_check['vehicle_age'] <= 100)

three = drivers_check[drivers_check['severity'] != 'Без пострадавших (нет данных)']
print(compare_groups(three, 'vehicle_age', 'severity').round(4))

### В2. Категории против одного столбца

Что сильнее связано с исходом: вид спора, суд, наличие адвоката? Для каждого столбца из списка — chi_square из Б5 против целевого столбца target. Строки таблицы — столбцы, по убыванию V Крамера: на больших данных p значимы почти у всех, а V показывает, какая связь сильнее. Порядок по V честен для столбцов с одинаковым числом значений: при независимости у столбца со многими значениями V не ноль, а около √(df / n), где df — степени свободы его таблицы, поэтому V таблиц разной формы сравнивают только у значимых связей. Пропуск в одном столбце убирает строку только из его таблицы. Если столбцов больше пяти, их p сравнивают с 0,05, делённым на число строк, — поправка Бонферрони.

Реализуйте функцию `category_tests(df, target, columns)`, которая проверяет хи-квадратом связь нескольких категориальных столбцов с одним целевым и упорядочивает их по силе связи. Аргументы: `df` — таблица с целевым столбцом target и столбцами из columns; `target` — имя целевого категориального столбца, строка; `columns` — список имён категориальных столбцов, строки. Возвращаемое значение: объект DataFrame: строка на каждый столбец из columns, индекс — имена столбцов по убыванию V; столбцы «хи-квадрат», «p», «V» — χ², p и V из chi_square(df, столбец, target). Таблица df не меняется.

Подсказка: строки — словарь `rows[column] = {'хи-квадрат': chi2, 'p': p, 'V': v}` в цикле по columns; таблица — `pd.DataFrame.from_dict(rows, orient='index')`; порядок — `sort_values('V', ascending=False)`. Сортировка по p ставит первым не самый сильно связанный столбец: при разном числе наблюдений и степеней свободы порядок p и V расходится; без `ascending=False` порядок обратный; `df.dropna()` по всей таблице выбрасывает строку с пропуском в одном столбце из таблиц всех остальных.

Проверка в ноутбуке: связь ДТП с погибшими с выходным днём, недостатками дороги, погодой и видом ДТП через category_tests; печатается таблица, в тест вводятся столбец с наибольшим V, его V до тысячных, p погоды до тысячных и решение по погоде.

In [ ]:
def category_tests(df, target, columns):
    """
    Проверяет хи-квадратом связь нескольких категориальных столбцов с одним
    целевым и упорядочивает их по силе связи.

    Что сильнее связано с исходом: вид спора, суд, наличие адвоката? Для каждого
    столбца из списка — chi_square из Б5 против целевого столбца target. Строки
    таблицы — столбцы, по убыванию V Крамера: на больших данных p значимы почти
    у всех, а V показывает, какая связь сильнее. Порядок по V честен для
    столбцов с одинаковым числом значений: при независимости у столбца со
    многими значениями V не ноль, а около √(df / n), где df — степени свободы
    его таблицы, поэтому V таблиц разной формы сравнивают только у значимых
    связей. Пропуск в одном столбце убирает строку только из его таблицы. Если
    столбцов больше пяти, их p сравнивают с 0,05, делённым на число строк, —
    поправка Бонферрони.

    Аргументы:
        df: Таблица с целевым столбцом target и столбцами из columns.
        target: Имя целевого категориального столбца, строка.
        columns: Список имён категориальных столбцов, строки.

    Возвращаемое значение:
        Объект DataFrame: строка на каждый столбец из columns, индекс — имена
        столбцов по убыванию V; столбцы «хи-квадрат», «p», «V» — χ², p и V из
        chi_square(df, столбец, target). Таблица df не меняется.
    """

    pass

In [ ]:
def category_tests_tests():
    df = pd.DataFrame({
        'исход': ['иск', 'отказ', 'иск', 'отказ', 'отказ', 'отказ', 'отказ', None, 'отказ', 'иск', 'отказ', 'отказ', 'отказ', 'отказ', 'отказ', 'иск', 'иск', 'иск', 'отказ', 'иск', 'иск', 'иск'],
        'адвокат': ['да', 'да', 'да', None, 'нет', 'нет', None, 'да', 'нет', 'да', 'да', 'да', 'да', None, 'да', 'да', 'да', 'да', 'нет', None, 'да', 'да'],
        'суд': ['А', 'Б', 'А', None, 'Б', 'А', 'Б', 'А', 'Б', 'А', 'Б', 'Б', 'Б', 'Б', 'Б', 'Б', 'А', 'А', 'Б', 'А', 'А', 'А'],
        'год': [2023, 2024, 2025, 2025, 2023, 2023, 2024, 2024, 2023, 2025, 2023, 2023, 2025, 2023, 2025, 2025, 2023, 2025, 2024, 2025, 2025, 2023],
        'сумма': [50.0, 120, None, 80, 45, None, 200, 65, 90, 150, None, 70, 110, 60, 85, None, 95, 130, 40, 75, 100, None],
    })
    before = df.copy()
    table = category_tests(df, 'исход', ['адвокат', 'суд', 'год'])
    assert isinstance(table, pd.DataFrame), 'функция должна возвращать DataFrame'
    assert table.columns.tolist() == ['хи-квадрат', 'p', 'V'], 'столбцы как в условии, в том же порядке'
    assert table.index.tolist() == ['суд', 'адвокат', 'год'], 'строки по убыванию V; по возрастанию p порядок был бы суд, год, адвокат: у адвоката четыре пропуска, таблица меньше'
    assert table.round(4).values.tolist() == [[9.7153, 0.0018, 0.798], [2.5075, 0.1133, 0.523], [4.6667, 0.097, 0.4714]], 'числа — chi_square(df, столбец, target): χ² и p у таблиц 2 × 2 с поправкой Йейтса, V без неё; строка с пропуском в столбце не входит только в его таблицу, пропуски суммы ни на что не влияют'
    assert df.equals(before), 'таблица изменилась'

    print('Все тесты прошли успешно!')

In [ ]:
category_tests_tests()

In [ ]:
# Проверка: связь ДТП с погибшими с выходным днём, недостатками дороги, погодой и видом ДТП через category_tests; печатается таблица, в тест вводятся столбец с наибольшим V, его V до тысячных, p погоды до тысячных и решение по погоде

df_check = pd.read_csv('../datasets/legal/dtp-moscow-region.csv', index_col='id', parse_dates=['datetime'])

flags = df_check.assign(fatal=df_check['dead_count'] > 0, weekend=df_check['datetime'].dt.dayofweek >= 5, defects=df_check['road_defects'] > 0)
print(category_tests(flags, 'fatal', ['weekend', 'defects', 'weather', 'category']).round(4))

### В3. Таблица гипотез курса

Шесть разных вопросов о ДТП, на каждый — один тест, одной функцией на любых таблицах той же структуры: другой области, другом периоде, части ДТП. Гипотеза 1 — стаж погибших и раненых водителей, t_two из Б1; 2 — число ДТП в районах в первый и последний год таблицы, парный t-тест `stats.ttest_rel` по таблице `pd.crosstab` района и года, где у района без ДТП в году ноль; строка, где вместо района записана сама область, — значение district со словом «область», ДТП без указания района, — в пары не входит; 3 — тёмное время суток и ДТП с погибшими, chi_square из Б5: тёмное время — значение light со словом «темное», ДТП с погибшими — dead_count больше нуля; 4 — стаж и возраст машины, r Пирсона из corr_test, Б7; 5 — стаж женщин и мужчин за рулём, t_two, женщины — первая группа: значение gender «Женский», мужчины — «Мужской»; 6 — недостатки дорожных условий и ДТП с погибшими, chi_square: недостатки установлены, если road_defects больше нуля.

Один вопрос — одна гипотеза: критерий Манна–Уитни на вопрос гипотезы 1 в таблицу не идёт, а нормальность — проверка условия теста, а не вопрос о деле, и в семейство поправки не входит. Таблица drivers — водители с их ДТП, как в разминке А1. Строки результата — как сводная таблица гипотез этапа 3: вопрос, тест, n, статистика, p, размер эффекта числом и словом через effect_label из Б8, решение при 0,05 и после поправки Бонферрони: гипотез шесть, порог 0,05 / 6.

Реализуйте функцию `hypotheses_report(df, drivers)`, которая собирает таблицу шести гипотез курса о ДТП и водителях — образец раздела гипотез отчёта. Аргументы: `df` — таблица ДТП со столбцами datetime — даты, district, light, dead_count и road_defects; `drivers` — таблица водителей со столбцами experience_clean, vehicle_age, health и gender. Возвращаемое значение: объект DataFrame из шести строк в порядке гипотез, индекс 1–6; столбцы «вопрос» — вопрос словами; «тест» — 't Уэлча', 'парный t', 'хи-квадрат', 'Пирсон', 't Уэлча', 'хи-квадрат'; «n» — погибшие и раненые водители без пропусков стажа, районы без строки области, ДТП таблицы df, пары без пропусков, женщины и мужчины без пропусков стажа, снова ДТП таблицы df; «статистика» — t, t, χ², r, t, χ²; «p»; «эффект» — d, nan, V, r, d, V; «размер» — слово effect_label для эффекта с его мерой, 'd', 'V' или 'r', у парного t — строка '—'; «решение» — «отвергаем», если p < 0,05, иначе «не отвергаем»; «после поправки» — то же при пороге 0,05 / 6. Таблицы не меняются.

Подсказка: строки удобно собрать списком кортежей `(вопрос, тест, n, статистика, p, эффект, размер)` и превратить в таблицу: `pd.DataFrame(rows, columns=['вопрос', 'тест', 'n', 'статистика', 'p', 'эффект', 'размер'], index=range(1, 7))`; решения — `np.where(table['p'] < 0.05, 'отвергаем', 'не отвергаем')` и то же с порогом `0.05 / len(table)`. Год — `df['datetime'].dt.year`, первый и последний — `min()` и `max()`; строку области убирает отбор `by_year[~by_year.index.str.contains('область')]`; флаги для chi_square — `df.assign(dark=..., fatal=..., defects=...)`. Пары районов через `pivot_table` и `dropna()` теряют районы без ДТП в одном из лет — у crosstab там ноль; `drop('Брянская область')` на таблицах другой области падает с `KeyError`; n гипотез 1 и 5 считается без пропусков стажа; пороги d для V и r дают не то слово.

Проверка в ноутбуке: таблица шести гипотез по ДТП 2024–2025 годов через hypotheses_report; печатается таблица без столбца «вопрос», в тест вводятся статистики гипотез 2 и 5 до сотых, p гипотезы 2 до тысячных и её решение, эффекты гипотез 1, 3 и 6 до тысячных, размер эффекта гипотезы 5 и число гипотез, отвергнутых после поправки.

In [ ]:
def hypotheses_report(df, drivers):
    """
    Собирает таблицу шести гипотез курса о ДТП и водителях — образец раздела
    гипотез отчёта.

    Шесть разных вопросов о ДТП, на каждый — один тест, одной функцией на любых
    таблицах той же структуры: другой области, другом периоде, части ДТП.
    Гипотеза 1 — стаж погибших и раненых водителей, t_two из Б1; 2 — число ДТП в
    районах в первый и последний год таблицы, парный t-тест `stats.ttest_rel` по
    таблице `pd.crosstab` района и года, где у района без ДТП в году ноль;
    строка, где вместо района записана сама область, — значение district со
    словом «область», ДТП без указания района, — в пары не входит; 3 — тёмное
    время суток и ДТП с погибшими, chi_square из Б5: тёмное время — значение
    light со словом «темное», ДТП с погибшими — dead_count больше нуля; 4 — стаж
    и возраст машины, r Пирсона из corr_test, Б7; 5 — стаж женщин и мужчин за
    рулём, t_two, женщины — первая группа: значение gender «Женский», мужчины —
    «Мужской»; 6 — недостатки дорожных условий и ДТП с погибшими, chi_square:
    недостатки установлены, если road_defects больше нуля.

    Один вопрос — одна гипотеза: критерий Манна–Уитни на вопрос гипотезы 1 в
    таблицу не идёт, а нормальность — проверка условия теста, а не вопрос о
    деле, и в семейство поправки не входит. Таблица drivers — водители с их ДТП,
    как в разминке А1. Строки результата — как сводная таблица гипотез этапа 3:
    вопрос, тест, n, статистика, p, размер эффекта числом и словом через
    effect_label из Б8, решение при 0,05 и после поправки Бонферрони: гипотез
    шесть, порог 0,05 / 6.

    Аргументы:
        df: Таблица ДТП со столбцами datetime — даты, district, light,
            dead_count и road_defects.
        drivers: Таблица водителей со столбцами experience_clean, vehicle_age,
                 health и gender.

    Возвращаемое значение:
        Объект DataFrame из шести строк в порядке гипотез, индекс 1–6; столбцы
        «вопрос» — вопрос словами; «тест» — 't Уэлча', 'парный t', 'хи-квадрат',
        'Пирсон', 't Уэлча', 'хи-квадрат'; «n» — погибшие и раненые водители без
        пропусков стажа, районы без строки области, ДТП таблицы df, пары без
        пропусков, женщины и мужчины без пропусков стажа, снова ДТП таблицы df;
        «статистика» — t, t, χ², r, t, χ²; «p»; «эффект» — d, nan, V, r, d, V;
        «размер» — слово effect_label для эффекта с его мерой, 'd', 'V' или 'r',
        у парного t — строка '—'; «решение» — «отвергаем», если p < 0,05, иначе
        «не отвергаем»; «после поправки» — то же при пороге 0,05 / 6. Таблицы не
        меняются.
    """

    pass

In [ ]:
def hypotheses_report_tests():
    day, lit, dusk, unlit = 'Светлое время суток', 'В темное время суток, освещение включено', 'Сумерки', 'В темное время суток, освещение отсутствует'
    df = pd.DataFrame({
        'datetime': pd.to_datetime(['2023-01-05 10:00', '2023-02-11 22:30', '2023-03-03 18:15', '2023-04-20 07:40', '2023-05-09 23:10', '2023-06-14 13:00',
                                    '2023-07-01 02:20', '2024-03-08 12:00', '2024-08-19 21:45', '2025-01-15 09:30', '2025-02-27 20:00', '2025-06-30 16:20',
                                    '2025-09-12 01:15', '2025-11-03 19:40', '2023-10-10 11:00', '2025-12-24 15:00', '2023-03-15 08:10', '2023-11-21 18:30',
                                    '2023-12-02 22:05']),
        'district': ['А', 'Б', 'В', 'Г', 'А', 'Б', 'В', 'А', 'Б', 'А', 'Б', 'В', 'В', 'А', 'Г', 'А', 'Калужская область', 'Калужская область', 'Калужская область'],
        'light': [day, lit, dusk, day, unlit, day, unlit, day, lit, day, unlit, day, unlit, lit, day, day, day, lit, unlit],
        'dead_count': [0, 1, 0, 0, 2, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 1],
        'road_defects': [0, 2, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 2, 0, 0, 0, 0, 1, 1],
    })
    drivers = pd.DataFrame({
        'experience_clean': [12.0, 3, 25, None, 8, 30, 15, 1, 20, 5, None, 40, 10, 7, 18, 22, 4, 9],
        'vehicle_age': [5.0, 12, 3, 8, None, 15, 9, 20, 4, 11, 6, None, 7, 10, 2, 13, 14, 6],
        'health': ['Погиб', 'Ранен', 'Погиб', 'Ранен', 'Ранен', 'Погиб', 'Не пострадал', 'Ранен', 'Не пострадал', 'Ранен', 'Погиб', 'Не пострадал',
                   'Ранен', 'Не пострадал', 'Погиб', None, 'Ранен', 'Не пострадал'],
        'gender': ['Мужской', 'Женский', 'Мужской', 'Мужской', 'Женский', 'Мужской', 'Мужской', 'Женский', 'Мужской', 'Женский', 'Мужской', 'Мужской',
                   'Мужской', 'Женский', None, 'Мужской', 'Женский', 'Мужской'],
    })
    before_df, before_drivers = df.copy(), drivers.copy()
    table = hypotheses_report(df, drivers)
    assert isinstance(table, pd.DataFrame), 'функция должна возвращать DataFrame'
    assert table.columns.tolist() == ['вопрос', 'тест', 'n', 'статистика', 'p', 'эффект', 'размер', 'решение', 'после поправки'], 'столбцы как в условии, в том же порядке'
    assert table.index.tolist() == [1, 2, 3, 4, 5, 6], 'шесть строк, индекс с единицы'
    assert table['тест'].tolist() == ['t Уэлча', 'парный t', 'хи-квадрат', 'Пирсон', 't Уэлча', 'хи-квадрат'], 'названия тестов — как в условии над заготовкой, их удобно скопировать оттуда'
    assert table['n'].tolist() == [10, 4, 19, 14, 15, 19], 'n: погибшие и раненые без пропусков стажа, районы без строки «Калужская область», ДТП, пары без пропусков, женщины и мужчины без пропусков стажа, ДТП'
    assert table['статистика'].round(4).tolist() == [3.8568, 0.7746, 4.3284, -0.349, -4.3875, 6.0461], 'статистики: t Уэлча, парный t по crosstab с нулём у района Г в 2025 году и без строки области, χ² с поправкой Йейтса, r Пирсона, t Уэлча женщин против мужчин, χ² флага недостатков'
    assert table['p'].round(4).tolist() == [0.0209, 0.495, 0.0375, 0.2213, 0.0016, 0.0139], 'p шести тестов'
    effect = table['эффект']
    assert effect.round(4).drop(2).tolist() == [2.6582, 0.5866, -0.349, -2.099, 0.6746], 'эффект: d Коэна, V Крамера по χ² без поправки Йейтса, r Пирсона, d Коэна, V Крамера'
    assert np.isnan(effect[2]), 'у парного t эффект — nan'
    assert table['размер'].tolist() == ['большой', '—', 'большой', 'средний', 'большой', 'большой'], 'размер — effect_label с мерой своей строки: d, V или r; у парного t — прочерк'
    assert table['решение'].tolist() == ['отвергаем', 'не отвергаем', 'отвергаем', 'не отвергаем', 'отвергаем', 'отвергаем'], 'решение при 0,05'
    assert table['после поправки'].tolist() == ['не отвергаем', 'не отвергаем', 'не отвергаем', 'не отвергаем', 'отвергаем', 'не отвергаем'], 'после поправки Бонферрони: порог 0,05 / 6'
    assert df.equals(before_df) and drivers.equals(before_drivers), 'таблицы изменились'

    print('Все тесты прошли успешно!')

In [ ]:
hypotheses_report_tests()

In [ ]:
# Проверка: таблица шести гипотез по ДТП 2024–2025 годов через hypotheses_report; печатается таблица без столбца «вопрос», в тест вводятся статистики гипотез 2 и 5 до сотых, p гипотезы 2 до тысячных и её решение, эффекты гипотез 1, 3 и 6 до тысячных, размер эффекта гипотезы 5 и число гипотез, отвергнутых после поправки

df_check = pd.read_csv('../datasets/legal/dtp-moscow-region.csv', index_col='id', parse_dates=['datetime'])
people_check = pd.read_csv('../datasets/legal/dtp-moscow-region-participants.csv')
people_check['experience_clean'] = people_check['experience'].replace([95, 96, 99], np.nan)
drivers_check = people_check[people_check['role'] == 'Водитель'].merge(df_check, left_on='dtp_id', right_index=True)
drivers_check['vehicle_age'] = drivers_check['datetime'].dt.year - drivers_check['vehicle_year']
drivers_check['vehicle_age'] = drivers_check['vehicle_age'].where(drivers_check['vehicle_age'] <= 100)

recent = df_check[df_check['datetime'].dt.year >= 2024]
recent_drivers = drivers_check[drivers_check['datetime'].dt.year >= 2024]
print(hypotheses_report(recent, recent_drivers).drop(columns='вопрос').round(3))

## Необязательная задача. Гипотезы 3 и 4 этапа 3 для своего датасета

Этап 3 проекта сдаётся на семинаре 14: не меньше четырёх гипотез разных видов на своём датасете, каждая записана по формату семинара 12 — вопрос, H0 и H1, тест и почему, n, статистика и p, интервал или размер эффекта, вывод словами с оговоркой о практической значимости. Гипотезы 1 и 2 — нормальность и одна выборка — делались в необязательных задачах семинаров 11 и 12. Оставшиеся две — сегодняшние; записанные по формату, они переходят в отчёт этапа 3 как есть.

Гипотеза 3 — сравнение числового столбца в двух группах: `t_two` или `mw_two`, тест выбирается по нормальности из гипотезы 1 и размеру групп. В записи — размеры групп, средние или медианы, d Коэна и его размер словами через `effect_label`, ящики двух групп с заголовком и подписями осей и оговорка: связь — не причина. Гипотеза 4 — связь двух категориальных столбцов: таблица долей по строкам, `pd.crosstab(..., normalize='index')`, тест `chi_square` или `fisher_2x2` для малой таблицы 2 × 2, V Крамера и вывод словами по долям; если в части клеток ожидаемых меньше пяти, редкие категории объединяются.

Если в датасете есть пары — одни и те же суды, регионы или люди в два момента, — можно добавить парное сравнение, `paired_test`: в этапе 3 это пятая гипотеза и бонус. Больше пяти гипотез — поправка Бонферрони.

Если датасета ещё нет, возьмите таблицы Брянской области `datasets/legal/dtp-bryansk.csv` и `datasets/legal/dtp-bryansk-participants.csv` и вопросы, которых нет в ноутбуке занятия: не стаж погибших и раненых водителей или женщин и мужчин, не возраст машины у женщин и мужчин, не тёмное время, не погода и не недостатки дороги.

В ответ в тест Moodle вставьте код и записи обеих гипотез; вложений у этого вопроса нет. Балл ставит преподаватель; автоматической проверки нет.

In [ ]:
path = '../datasets/legal/dtp-bryansk.csv'   # путь к своему файлу; пока таблица ДТП Брянской области
data = pd.read_csv(path)

print(data.shape)
print(data.dtypes)

In [ ]:
# гипотеза 3: сравнение двух групп

In [ ]:
# гипотеза 4: связь двух категориальных столбцов

Записи гипотез

| Пункт | Гипотеза 3 | Гипотеза 4 |
|---|---|---|
| вопрос | | |
| H0 и H1 | | |
| тест и почему | | |
| n | | |
| статистика и p | | |
| интервал или размер эффекта | | |
| вывод | | |